In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_recall_curve, f1_score

In [ ]:
# 1. Loading the cleaned dataset
df = pd.read_csv('data/processed/cleaned_diabetic_data.csv')

dead_hospice_ids = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(dead_hospice_ids)]

# Separating target from features
y = df['target']
X = df.drop(columns=['target', 'readmitted'])

# 2. Defining column types
cat_cols = [
    # Demographics & Admin
    'race', 'gender', 'age', 'payer_code', 'medical_specialty', 
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    
    # Diagnostics & General Meds
    'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 
    'change', 'diabetesMed',
    
    # Specific Medications
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 
    'miglitol', 'troglitazone', 'tolazamide', 'insulin', 
    'glyburide-metformin', 'glipizide-metformin', 
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 
    'metformin-pioglitazone'
]

# The remaining columns (like time_in_hospital, num_lab_procedures, etc.) will now correctly be numerical
num_cols = [col for col in X.columns if col not in cat_cols]

# Clean categorical data types for XGBoost
for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].astype(str).astype('category')

X_xgb = X.copy()
# XGBoost needs the categoricals cast to the 'category' dtype
existing_cats = [col for col in cat_cols if col in X.columns]
X_xgb[existing_cats] = X_xgb[existing_cats].astype('category')

X_lr = X.copy()
# 1. One-Hot Encoding categoricals (drop_first=True prevents multicollinearity)
X_lr = pd.get_dummies(X_lr, columns=existing_cats, drop_first=True, dummy_na=True)

# XGBoost Split
X_train_xgb, X_test_xgb, y_train, y_test = train_test_split(X_xgb, y, test_size=0.2, random_state=42, stratify=y)

# Logistic Regression Split
X_train_lr, X_test_lr, _, _ = train_test_split(X_lr, y, test_size=0.2, random_state=42, stratify=y)

# 2. Scaling the numerical columns for Logistic Regression for data leakage
scaler = StandardScaler()
X_train_lr[num_cols] = scaler.fit_transform(X_train_lr[num_cols])
X_test_lr[num_cols] = scaler.transform(X_test_lr[num_cols])

print(f"Filtered Dataset Size: {df.shape[0]} rows")
print(f"XGBoost Training Features: {X_train_xgb.shape[1]}")
print(f"Logistic Regression Training Features: {X_train_lr.shape[1]}")

Filtered Dataset Size: 68150 rows
XGBoost Training Features: 43
Logistic Regression Training Features: 263


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Initializing and fitting the baseline model
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_lr, y_train)

# Predict on test set
y_pred_lr = lr_model.predict(X_test_lr)
y_prob_lr = lr_model.predict_proba(X_test_lr)[:, 1]

# Print core evaluation metrics
print("--- Logistic Regression Baseline Results ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr):.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

# 5-Fold Cross-Validation for robust performance checks
cv_scores = cross_val_score(lr_model, X_train_lr, y_train, cv=5, scoring='f1')
print(f"5-Fold CV F1-Scores: {cv_scores}")
print(f"Mean CV F1-Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

--- Logistic Regression Baseline Results ---
ROC-AUC Score: 0.6626

Confusion Matrix:
[[12694     7]
 [ 1229     4]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      1.00      0.95     12701
           1       0.36      0.00      0.01      1233

    accuracy                           0.91     13934
   macro avg       0.64      0.50      0.48     13934
weighted avg       0.86      0.91      0.87     13934

5-Fold CV F1-Scores: [0.01986097 0.00604839 0.01001001 0.0080402  0.01192843]
Mean CV F1-Score: 0.0112 (+/- 0.0048)


In [ ]:
# Initialize with class weighting to handle severe imbalance
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train_lr, y_train)

# Re-run predictions
y_pred_lr = lr_model.predict(X_test_lr)
y_prob_lr = lr_model.predict_proba(X_test_lr)[:, 1]

# Check updated metrics
print("--- Logistic Regression (Balanced) Results ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr):.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

--- Logistic Regression (Balanced) Results ---
ROC-AUC Score: 0.6626

Confusion Matrix:
[[8423 4278]
 [ 555  678]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.66      0.78     12701
           1       0.14      0.55      0.22      1233

    accuracy                           0.65     13934
   macro avg       0.54      0.61      0.50     13934
weighted avg       0.87      0.65      0.73     13934



In [ ]:
# Extract coefficients and map them to feature names
coef_df = pd.DataFrame({
    'Feature': X_train_lr.columns,
    'Coefficient': lr_model.coef_[0]
})

# Sort by absolute value to find the strongest drivers (positive and negative)
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)

print("--- Top 10 Strongest Logistic Regression Predictors ---")
print(coef_df[['Feature', 'Coefficient']].head(10).to_string(index=False))

--- Top 10 Strongest Logistic Regression Predictors ---
                                   Feature  Coefficient
               discharge_disposition_id_11    -4.712343
medical_specialty_Pediatrics-Endocrinology    -2.584527
               discharge_disposition_id_15     2.058460
 medical_specialty_Pediatrics-CriticalCare    -1.897645
               discharge_disposition_id_28     1.844424
              medical_specialty_Gynecology    -1.833024
               discharge_disposition_id_22     1.645899
               discharge_disposition_id_13    -1.438233
          medical_specialty_Otolaryngology    -1.367794
               discharge_disposition_id_23    -1.201083


In [ ]:
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

tuned_xgb_model = xgb.XGBClassifier(
    enable_categorical=True,
    scale_pos_weight=scale_pos_weight,
    max_depth=4,                # Shallower trees to prevent overfitting
    learning_rate=0.05,         # Slower learning rate for better convergence
    n_estimators=200,           # More boosting rounds to compensate for lower learning rate
    subsample=0.8,              # Row subsampling for regularization
    colsample_bytree=0.8,       # Feature subsampling
    random_state=42,
    eval_metric='logloss'
)

tuned_xgb_model.fit(X_train_xgb, y_train)

# Evaluate tuned performance
y_pred_tuned = tuned_xgb_model.predict(X_test_xgb)
y_prob_tuned = tuned_xgb_model.predict_proba(X_test_xgb)[:, 1]

print("--- Tuned XGBoost Model Results ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_tuned):.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

--- Tuned XGBoost Model Results ---
ROC-AUC Score: 0.6684

Confusion Matrix:
[[8706 3995]
 [ 545  688]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.69      0.79     12701
           1       0.15      0.56      0.23      1233

    accuracy                           0.67     13934
   macro avg       0.54      0.62      0.51     13934
weighted avg       0.87      0.67      0.74     13934



In [ ]:
# Convert all defined category columns to string first to strip out any hidden floats, then to category
for col in cat_cols:
    if col in X.columns:
        X[col] = X[col].astype(str).astype('category')

# Re-run your XGBoost split with clean categories
X_xgb = X.copy()
X_train_xgb, X_test_xgb, y_train, y_test = train_test_split(
    X_xgb, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_cv_scores = []

for train_idx, val_idx in skf.split(X_xgb, y):
    X_tr, X_va = X_xgb.iloc[train_idx], X_xgb.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    scale_pos_weight_fold = (len(y_tr) - sum(y_tr)) / sum(y_tr)
    fold_xgb = xgb.XGBClassifier(
        enable_categorical=True,
        scale_pos_weight=scale_pos_weight_fold,
        max_depth=4,
        learning_rate=0.05,
        n_estimators=200,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    )
    fold_xgb.fit(X_tr, y_tr)
    preds = fold_xgb.predict(X_va)
    xgb_cv_scores.append(f1_score(y_va, preds))

print(f"XGBoost 5-Fold CV F1-Scores: {np.round(xgb_cv_scores, 4)}")
print(f"XGBoost Mean CV F1-Score: {np.mean(xgb_cv_scores):.4f} (+/- {np.std(xgb_cv_scores):.4f})")

XGBoost 5-Fold CV F1-Scores: [0.2388 0.2297 0.2229 0.2214 0.233 ]
XGBoost Mean CV F1-Score: 0.2292 (+/- 0.0065)


In [ ]:
#Precision Recall Curve Analysis for Tuned XGBoost Model
y_prob_tuned = tuned_xgb_model.predict_proba(X_test_xgb)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_tuned)

# Target ~60% recall operating point
target_recall_idx = np.argmin(np.abs(recalls - 0.60))
custom_threshold = thresholds[target_recall_idx]
custom_preds = (y_prob_tuned >= custom_threshold).astype(int)

print(f"Selected Decision Threshold: {custom_threshold:.4f}")
print(f"Achieved Recall:    {recalls[target_recall_idx]:.4f}")
print(f"Achieved Precision: {precisions[target_recall_idx]:.4f}\n")
print("Confusion Matrix at Custom Threshold:")
print(confusion_matrix(y_test, custom_preds))
print("\nClassification Report at Custom Threshold:")
print(classification_report(y_test, custom_preds))

Selected Decision Threshold: 0.5066
Achieved Recall:    0.6000
Achieved Precision: 0.1676

Confusion Matrix at Custom Threshold:
[[8734 3666]
 [ 492  738]]

Classification Report at Custom Threshold:
              precision    recall  f1-score   support

           0       0.95      0.70      0.81     12400
           1       0.17      0.60      0.26      1230

    accuracy                           0.69     13630
   macro avg       0.56      0.65      0.53     13630
weighted avg       0.88      0.69      0.76     13630

